# Statevectors, gates, and global phase

Exercise common one- and two-qubit gates, Qiskit's little-endian ordering, and global phase.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

This circuit mixes rotations, controlled phases, XX/YY/ZZ interactions, and a global phase to exercise Qiskit's gate and endian conventions.

In [2]:
circuit = QuantumCircuit(3, name="gate-parity")
circuit.h(0)
circuit.ry(0.37, 1)
circuit.cx(0, 2)
circuit.cp(-0.23, 2, 1)
circuit.rxx(0.41, 0, 1)
circuit.ryy(-0.19, 1, 2)
circuit.rzz(0.29, 2, 0)
circuit.global_phase = 0.17

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(
    lambda: np.asarray(Statevector.from_instruction(circuit).data)
)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def run_mettleq():
    return np.asarray(backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"])

candidate, mettleq_ms, _ = benchmark(run_mettleq)
error = phase_aligned_statevector_error(reference, candidate)
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

The complete statevector is phase-aligned and compared amplitude by amplitude; its norm is checked separately.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/02_statevectors_and_gates.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase-aligned statevector atol=2e-6",
    passed=error <= 2e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_amplitude_error": error, "norm": float(np.linalg.norm(candidate))},
)


Comparison summary
------------------
Correctness contract: PASS — phase-aligned statevector atol=2e-6
SDK reference median: 0.159 ms
MettleQ median:       0.930 ms
Timing interpretation: the SDK reference was 5.843x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "phase-aligned statevector atol=2e-6", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"max_amplitude_error": 4.350176504536388e-08, "norm": 1.0}, "mettleq_median_ms": 0.9304590057581663, "notebook": "qiskit/02_statevectors_and_gates.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 0.15924999024719, "reference_over_mettleq": 0.17115207576225056, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

Full-state readback is useful for debugging and small exact studies, but its transfer cost matters at larger widths.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.